# PIEFS — Classical Baselines (GPU session)

> **Recommended:** Runtime → Change runtime type → **T4 GPU**  
> sklearn methods compute on CPU cores (GPU doesn't help them), but a GPU session
> gives you faster CPU and more RAM on Colab — useful for KernelPCA on MNIST/CIFAR.

Compares PIEFS against classical embedding and classification methods on all 7 benchmark datasets.

**Methods:** PCA, KernelPCA (rbf/poly/cosine), TruncSVD, FastICA, Gaussian Random Projection,
Nyström, SpectralEmbedding, LDA, RandomForest, SVM (rbf), raw LogisticRegression

**Metrics:** accuracy, ROC-AUC, macro-F1, MCC, precision, recall, gram_error (‖C−I‖²_F), fit_time

**Datasets:** Two Moons, Circles, HTRU2, MNIST-bin, MNIST-10, CIFAR-bin, CIFAR-10 features

Results are saved incrementally to Google Drive `PIEFS_baselines/` and downloaded as a ZIP.

In [ ]:
!pip install -q scikit-learn numpy torch torchvision tqdm

In [ ]:
# ── Runtime info ───────────────────────────────────────────────────────────
import torch, os

if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    print(f'GPU available: {gpu.name}  ({gpu.total_memory/1e9:.1f} GB)')
    print('Note: sklearn methods run on CPU cores regardless of GPU.')
    print('      GPU session gives faster CPU + more RAM on Colab.')
else:
    print('No GPU detected — running on CPU only.')
    print('Consider: Runtime → Change runtime type → T4 GPU')

import multiprocessing
print(f'CPU cores available for sklearn: {multiprocessing.cpu_count()}')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/PIEFS_baselines'
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f'Drive output dir: {DRIVE_DIR}')

In [ ]:
# ── Data setup (no GitHub required) ───────────────────────────────────────
import os
from pathlib import Path

DATA_DIR = Path('/content/data')
DATA_DIR.mkdir(exist_ok=True)

# HTRU2: download from UCI archive (auto if not cached)
HTRU2_CSV = DATA_DIR / 'HTRU_2.csv'
if not HTRU2_CSV.exists():
    import urllib.request, zipfile, io
    print('Downloading HTRU2 from UCI...')
    url = 'https://archive.ics.uci.edu/static/public/372/htru2.zip'
    with urllib.request.urlopen(url) as r:
        zdata = r.read()
    with zipfile.ZipFile(io.BytesIO(zdata)) as zf:
        name = [n for n in zf.namelist() if n.endswith('.csv')][0]
        zf.extract(name, DATA_DIR)
        extracted = DATA_DIR / name
        extracted.rename(HTRU2_CSV)
    print(f'HTRU2 saved → {HTRU2_CSV}')
else:
    print(f'HTRU2 already cached: {HTRU2_CSV}')

print('Data setup complete.')

In [ ]:
from __future__ import annotations
import json, time, warnings, zipfile, shutil
from pathlib import Path

import numpy as np
from tqdm.notebook import tqdm

from sklearn.decomposition import KernelPCA, PCA, TruncatedSVD, FastICA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.ensemble import RandomForestClassifier
from sklearn.kernel_approximation import Nystroem
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, f1_score, matthews_corrcoef,
    precision_score, recall_score, roc_auc_score
)
from sklearn.pipeline import Pipeline
from sklearn.random_projection import GaussianRandomProjection
from sklearn.svm import SVC

warnings.filterwarnings('ignore')
print('Imports OK')

In [ ]:
# ── Configuration ──────────────────────────────────────────────────────────

SEEDS  = [0, 1, 2, 3, 4]
N_COMP = 6   # matches PIEFS K=6 (binary); loop uses 10 for multiclass

DATASETS_TO_RUN = [
    'two_moon',
    'circles',
    'htru2',
    'mnist_binary',
    'mnist_mc',
    'cifar10_binary',
    'cifar10_mc',
]

OUT_DIR        = Path('/content/results_baselines')
OUT_DIR.mkdir(exist_ok=True)
DRIVE_DIR_PATH = Path(DRIVE_DIR)

print(f'Seeds: {SEEDS},  n_components: {N_COMP}')

In [ ]:
# ── Dataset loaders (self-contained, no PIEFS source needed) ─────────────

import numpy as np
import torch
from sklearn.datasets import make_moons, make_circles
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

def _standardize(X_tr, X_te):
    sc = StandardScaler().fit(X_tr)
    return sc.transform(X_tr).astype(np.float32), sc.transform(X_te).astype(np.float32)


def load_two_moon(seed):
    X, y = make_moons(n_samples=10_000, noise=0.1, random_state=0)
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=seed)
    X_tr, X_te = _standardize(X_tr, X_te)
    return X_tr, y_tr, X_te, y_te


def load_circles(seed):
    X, y = make_circles(n_samples=10_000, noise=0.05, random_state=0)
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=seed)
    X_tr, X_te = _standardize(X_tr, X_te)
    return X_tr, y_tr, X_te, y_te


def load_htru2(seed):
    data = np.loadtxt(str(HTRU2_CSV), delimiter=',')
    X, y = data[:, :8].astype(np.float32), data[:, 8].astype(int)
    rng = np.random.default_rng(42)
    idx = rng.permutation(len(X))
    X, y = X[idx], y[idx]
    n_train = int(len(X) * 0.7)
    X_tr, y_tr = X[:n_train], y[:n_train]
    X_te, y_te = X[n_train:], y[n_train:]
    X_tr, X_te = _standardize(X_tr, X_te)
    return X_tr, y_tr, X_te, y_te


def _load_torchvision(name, task, binary_classes=(0, 1), val_fraction=0.1, seed=42):
    import torchvision.datasets as tvd
    import torchvision.transforms as T
    to_tensor = T.ToTensor()
    root = str(DATA_DIR / name)
    if name == 'mnist':
        tr = tvd.MNIST(root, train=True,  download=True, transform=to_tensor)
        te = tvd.MNIST(root, train=False, download=True, transform=to_tensor)
    elif name == 'cifar10':
        tr = tvd.CIFAR10(root, train=True,  download=True, transform=to_tensor)
        te = tvd.CIFAR10(root, train=False, download=True, transform=to_tensor)
    else:
        raise ValueError(name)

    def to_numpy(ds):
        loader = torch.utils.data.DataLoader(ds, batch_size=1024, shuffle=False)
        xs, ys = [], []
        for xb, yb in loader:
            xs.append(xb.flatten(1).float().numpy())
            ys.append(yb.numpy())
        return np.vstack(xs), np.concatenate(ys)

    X_tr_all, y_tr_all = to_numpy(tr)
    X_te,     y_te     = to_numpy(te)

    if task == 'binary':
        mask_tr = (y_tr_all == binary_classes[0]) | (y_tr_all == binary_classes[1])
        mask_te = (y_te     == binary_classes[0]) | (y_te     == binary_classes[1])
        X_tr_all, y_tr_all = X_tr_all[mask_tr], (y_tr_all[mask_tr] == binary_classes[1]).astype(int)
        X_te,     y_te     = X_te[mask_te],     (y_te[mask_te]     == binary_classes[1]).astype(int)

    # carve val → use train slice only for standardization
    rng2 = np.random.default_rng(seed)
    perm = rng2.permutation(len(X_tr_all))
    n_val = int(len(X_tr_all) * val_fraction)
    train_idx = perm[n_val:]
    X_tr = X_tr_all[train_idx]
    y_tr = y_tr_all[train_idx]

    sc = StandardScaler().fit(X_tr)
    X_tr = sc.transform(X_tr).astype(np.float32)
    X_te = sc.transform(X_te).astype(np.float32)
    return X_tr, y_tr, X_te, y_te


def load_mnist_binary(seed):
    return _load_torchvision('mnist', 'binary')

def load_mnist_mc(seed):
    return _load_torchvision('mnist', 'multiclass')

def load_cifar10_binary(seed):
    return _load_torchvision('cifar10', 'binary')

def load_cifar10_mc(seed):
    return _load_torchvision('cifar10', 'multiclass')


LOADERS = {
    'two_moon':       (load_two_moon,     'binary'),
    'circles':        (load_circles,      'binary'),
    'htru2':          (load_htru2,        'binary'),
    'mnist_binary':   (load_mnist_binary, 'binary'),
    'mnist_mc':       (load_mnist_mc,     'multiclass'),
    'cifar10_binary': (load_cifar10_binary, 'binary'),
    'cifar10_mc':     (load_cifar10_mc,   'multiclass'),
}
print('Loaders registered:', list(LOADERS))

In [ ]:
# ── Metrics ────────────────────────────────────────────────────────────────

def compute_all_metrics(y_true, y_pred, y_proba, features, task):
    avg = 'binary' if task == 'binary' else 'macro'
    acc = float(accuracy_score(y_true, y_pred))
    f1  = float(f1_score(y_true, y_pred, average=avg, zero_division=0))
    mcc = float(matthews_corrcoef(y_true, y_pred))
    pre = float(precision_score(y_true, y_pred, average=avg, zero_division=0))
    rec = float(recall_score(y_true, y_pred, average=avg, zero_division=0))
    try:
        p = y_proba[:,1] if task == 'binary' else y_proba
        auc = float(roc_auc_score(y_true, p,
                                   multi_class='ovr' if task == 'multiclass' else 'raise',
                                   average='macro'))
    except Exception:
        auc = float('nan')

    # gram_error = ‖C − I‖²_F  (same metric as PIEFS val_gram_error)
    gram_error = float('nan')
    if features is not None:
        try:
            F = features.astype(np.float64)
            C = F.T @ F / len(F)
            gram_error = float(np.sum((C - np.eye(C.shape[0])) ** 2))
        except Exception:
            pass

    return dict(acc=acc, auc=auc, f1_macro=f1, mcc=mcc,
                precision_macro=pre, recall_macro=rec, gram_error=gram_error)

print('Metrics function defined')

In [ ]:
# ── Pipeline builders ──────────────────────────────────────────────────────

def build_pipelines(n_comp, n_features, task, n_classes):
    nc  = min(n_comp, n_features - 1, 50)
    clf = (LogisticRegression(solver='lbfgs', max_iter=2000, C=1.0,
                               multi_class='multinomial', random_state=42)
           if task == 'multiclass' else
           LogisticRegression(solver='lbfgs', max_iter=2000, C=1.0, random_state=42))

    def with_pre(steps):
        if n_features > 100:
            pre = [('pre_pca', PCA(n_components=min(50, n_features-1), random_state=42))]
            return pre + steps
        return steps

    ps = {
        'LogReg_raw':      Pipeline([('clf', clf)]),
        'PCA+LogReg':      Pipeline([('pca', PCA(nc, random_state=42)), ('clf', clf)]),
        'TruncSVD+LogReg': Pipeline([('svd', TruncatedSVD(nc, random_state=42)), ('clf', clf)]),
        'GRP+LogReg':      Pipeline([('grp', GaussianRandomProjection(nc, random_state=42)), ('clf', clf)]),
        'FastICA+LogReg':  Pipeline([('ica', FastICA(nc, random_state=42, max_iter=500, tol=0.01)), ('clf', clf)]),
        'Nystroem+LogReg': Pipeline(with_pre([('ny', Nystroem('rbf', n_components=min(nc*8,300), random_state=42)), ('clf', clf)])),
        'KPCA_rbf+LogReg': Pipeline(with_pre([('kpca', KernelPCA(nc, kernel='rbf', random_state=42)), ('clf', clf)])),
        'KPCA_poly+LogReg':Pipeline(with_pre([('kpca', KernelPCA(nc, kernel='poly', degree=3, random_state=42)), ('clf', clf)])),
        'KPCA_cos+LogReg': Pipeline(with_pre([('kpca', KernelPCA(nc, kernel='cosine', random_state=42)), ('clf', clf)])),
        'RandomForest':    Pipeline([('rf', RandomForestClassifier(200, min_samples_leaf=2, random_state=42, n_jobs=-1))]),
        'SVM_rbf':         'svm_placeholder',
        'SpectralEmb+LogReg': 'spectral_placeholder',
    }
    lda_nc = min(n_classes - 1, n_features - 1)
    if lda_nc >= 1:
        ps['LDA+LogReg'] = Pipeline([('lda', LinearDiscriminantAnalysis(lda_nc)), ('clf', clf)])
    return ps

print('Pipeline builder defined')

In [ ]:
# ── Special methods (SVM, SpectralEmbedding) ───────────────────────────────

def run_svm(X_tr, y_tr, X_te, y_te, n_tr_max=10_000):
    if len(X_tr) > n_tr_max:
        idx = np.random.choice(len(X_tr), n_tr_max, replace=False)
        X_tr, y_tr = X_tr[idx], y_tr[idx]
    svm = SVC(kernel='rbf', probability=True, C=1.0, random_state=42)
    t0 = time.time(); svm.fit(X_tr, y_tr); fit_t = time.time() - t0
    t0 = time.time()
    return svm.predict(X_te), svm.predict_proba(X_te), fit_t, time.time()-t0, None


def run_spectral(X_tr, y_tr, X_te, y_te, n_comp, clf):
    from sklearn.manifold import SpectralEmbedding
    MAX_N = 5000
    N_tr, N_te = len(X_tr), len(X_te)
    if N_tr + N_te > MAX_N:
        n_tr_s = min(N_tr, MAX_N * N_tr // (N_tr + N_te))
        idx_tr = np.random.choice(N_tr, n_tr_s, replace=False)
        idx_te = np.random.choice(N_te, MAX_N - n_tr_s, replace=False)
        X_all = np.vstack([X_tr[idx_tr], X_te[idx_te]])
        y_tr_s, y_te_s = y_tr[idx_tr], y_te[idx_te]
    else:
        X_all = np.vstack([X_tr, X_te])
        y_tr_s, y_te_s, n_tr_s = y_tr, y_te, N_tr

    t0 = time.time()
    Z = SpectralEmbedding(n_components=n_comp, random_state=42, n_jobs=-1).fit_transform(X_all)
    fit_t = time.time() - t0
    Z_tr, Z_te = Z[:n_tr_s], Z[n_tr_s:]
    clf2 = type(clf)(**clf.get_params())
    clf2.fit(Z_tr, y_tr_s)
    return clf2.predict(Z_te), clf2.predict_proba(Z_te), fit_t, 0.0, Z_te, y_te_s

print('Special methods defined')

In [ ]:
# ── Main experiment loop ───────────────────────────────────────────────────

all_results = {}
resume_path = OUT_DIR / 'baselines_results.json'
if resume_path.exists():
    with open(resume_path) as f:
        all_results = json.load(f)
    print(f'Resumed from {resume_path}')

for ds_name in DATASETS_TO_RUN:
    loader_fn, task = LOADERS[ds_name]
    print(f'\n{"="*70}\nDATASET: {ds_name}  (task={task})\n{"="*70}')
    all_results.setdefault(ds_name, {})

    for seed in tqdm(SEEDS, desc=ds_name):
        np.random.seed(seed)
        try:
            X_tr, y_tr, X_te, y_te = loader_fn(seed)
        except Exception as e:
            print(f'  [SKIP] seed={seed}: {e}'); continue

        n_tr, n_fe = X_tr.shape
        n_classes  = len(np.unique(y_tr))
        n_comp     = 10 if task == 'multiclass' else N_COMP
        print(f'  seed={seed}: train={n_tr}, test={len(X_te)}, d={n_fe}, C={n_classes}')

        pipes    = build_pipelines(n_comp, n_fe, task, n_classes)
        lr_spec  = LogisticRegression(solver='lbfgs', max_iter=2000, random_state=42+seed)

        for name, pipe in pipes.items():
            if str(seed) in all_results[ds_name].get(name, {}):
                continue  # already done
            try:
                feats, y_eval = None, y_te
                if pipe == 'svm_placeholder':
                    yp, ypr, fit_t, inf_t, feats = run_svm(X_tr, y_tr, X_te, y_te)
                elif pipe == 'spectral_placeholder':
                    yp, ypr, fit_t, inf_t, feats, y_eval = run_spectral(X_tr, y_tr, X_te, y_te, n_comp, lr_spec)
                else:
                    t0 = time.time(); pipe.fit(X_tr, y_tr); fit_t = time.time()-t0
                    t0 = time.time()
                    yp  = pipe.predict(X_te)
                    ypr = pipe.predict_proba(X_te) if hasattr(pipe,'predict_proba') else None
                    inf_t = time.time()-t0
                    steps = pipe.steps
                    if len(steps) > 1 and hasattr(steps[-2][1], 'transform'):
                        from sklearn.pipeline import Pipeline as _P
                        feats = _P(steps[:-1]).transform(X_te)

                m = compute_all_metrics(y_eval, yp, ypr, feats, task)
                m.update(dict(fit_time=round(fit_t,3), inf_time=round(inf_t,3),
                              n_components=n_comp, n_train=n_tr, n_test=len(X_te), input_dim=n_fe))
                all_results[ds_name].setdefault(name, {})[str(seed)] = m
                print(f'    {name:<25} acc={m["acc"]:.4f} auc={m["auc"]:.4f} '
                      f'f1={m["f1_macro"]:.4f} mcc={m["mcc"]:.4f} '
                      f'gram={m["gram_error"]:.4f} t={fit_t:.1f}s')
            except Exception as exc:
                print(f'    {name:<25} ERROR: {exc}')

        with open(resume_path,'w') as f: json.dump(all_results,f,indent=2)
        shutil.copy(resume_path, DRIVE_DIR_PATH/'baselines_results.json')

print('\nAll experiments complete.')

In [ ]:
# ── Results table ──────────────────────────────────────────────────────────
import statistics

def summarize(results, metric='acc'):
    datasets = list(results.keys())
    methods  = []
    for ds in datasets:
        for m in results[ds]:
            if m not in methods: methods.append(m)
    col_w = 25
    hdr = f'{"Method":<{col_w}}' + ''.join(f'{d[:12]:>16}' for d in datasets)
    print(f'\n=== {metric.upper()} (mean±std) ===')
    print(hdr); print('-'*len(hdr))
    for method in methods:
        row = f'{method:<{col_w}}'
        for ds in datasets:
            vals = [results[ds].get(method,{}).get(str(s),{}).get(metric) for s in SEEDS]
            vals = [v for v in vals if v is not None and v==v]
            if vals:
                mv,sv = statistics.mean(vals), statistics.stdev(vals) if len(vals)>1 else 0
                sc = 100 if metric in ('acc','f1_macro','precision_macro','recall_macro') else 1
                row += f'{mv*sc:>9.2f}±{sv*sc:.2f}'
            else:
                row += f'{"—":>16}'
        print(row)

for m in ['acc','auc','f1_macro','mcc','gram_error']:
    summarize(all_results, m)

In [ ]:
# ── Save to Drive + download ZIP ───────────────────────────────────────────
from google.colab import files

shutil.copy(resume_path, DRIVE_DIR_PATH/'baselines_results.json')

table_path = OUT_DIR/'baselines_table.txt'
with open(table_path,'w') as f:
    import io, contextlib
    for m in ['acc','auc','f1_macro','mcc','gram_error','fit_time']:
        buf = io.StringIO()
        with contextlib.redirect_stdout(buf): summarize(all_results, m)
        f.write(buf.getvalue())
shutil.copy(table_path, DRIVE_DIR_PATH/'baselines_table.txt')

zip_path = '/content/piefs_baselines.zip'
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as zf:
    for p in OUT_DIR.rglob('*'):
        if p.is_file(): zf.write(p, p.relative_to(OUT_DIR))

shutil.copy(zip_path, DRIVE_DIR_PATH/'piefs_baselines.zip')
print(f'ZIP → Drive: {DRIVE_DIR_PATH}/piefs_baselines.zip')
files.download(zip_path)